# 01 · Imagen y vídeo con OpenCV 

**Por qué importa:** "conocimientos sólidos de tratamiento de imagen y vídeo" y
"experiencia práctica con OpenCV". 
1. Entiendes qué es una imagen en memoria (shape, dtype, BGR vs RGB, coordenadas).
2. Sabes el preprocesado que hace un detector (resize, *letterbox*, normalización).
3. Sabes leer y escribir vídeo y dónde se pierde el tiempo (decodificación).
4. Conoces técnicas clásicas que siguen siendo útiles: color (HSV), fondo (MOG2), flujo óptico.

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
ROOT = ROOT or Path.cwd()   # fuera del repo (Colab): los datos se guardan en ./data
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
from pathlib import Path
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO = DATA / "people.mp4"

def show(*imgs, titles=None, size=5, cmap=None):
    """Muestra imágenes BGR (o grises) con matplotlib."""
    fig, axs = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size))
    axs = np.atleast_1d(axs)
    for i, (ax, im) in enumerate(zip(axs, imgs)):
        ax.imshow(im if im.ndim == 2 else im[:, :, ::-1], cmap=cmap or ("gray" if im.ndim == 2 else None))
        ax.set_title(titles[i] if titles else "")
        ax.axis("off")
    plt.show()

cap = cv2.VideoCapture(str(VIDEO))
ok, frame = cap.read()
cap.release()
assert ok, "No se pudo leer el vídeo: ejecuta la celda SETUP"

## 1. Una imagen es un array de NumPy

- `shape = (H, W, C)` → **filas primero**. Un píxel se accede como `img[y, x]`, pero las funciones de
  OpenCV que reciben puntos usan `(x, y)`. Esta confusión es fuente clásica de bugs.
- `dtype = uint8` → valores 0–255. Si haces aritmética, convierte a `float32` o desbordará (250 + 10 = 4).
- OpenCV carga en **BGR**; matplotlib, PIL y casi todos los modelos esperan **RGB**.
- Los modelos PyTorch esperan **tensor `(N, C, H, W)` float en [0, 1]** (o normalizado con media/desviación).

In [ ]:
print("shape:", frame.shape, "dtype:", frame.dtype)
H, W = frame.shape[:2]
y, x = 100, 200
print("píxel (x=200, y=100) en BGR:", frame[y, x])

a = np.full((2, 2), 250, dtype=np.uint8)
print("uint8 overflow: 250 + 10 =", (a + 10)[0, 0], "| cv2.add satura:", cv2.add(a, np.full_like(a, 10))[0, 0])

# De OpenCV (H,W,C BGR uint8) a tensor de modelo (1,C,H,W RGB float):
import torch
t = torch.from_numpy(frame[:, :, ::-1].copy()).permute(2, 0, 1).float().div(255).unsqueeze(0)
print("tensor:", t.shape, t.dtype, float(t.min()), float(t.max()))

## 2. Espacios de color: HSV para segmentar por color

En HSV, **H** (tono) separa el color de la iluminación (**V**). Por eso es mucho más robusto que RGB para
cosas como "separar jugadores por color de camiseta" o detectar el césped. En OpenCV, H va de 0 a 179.

Uso real en deporte: recortar el torso de cada jugador detectado → histograma de color en HSV →
k-means con k=2 → equipo A / equipo B. (Lo haremos en el notebook 07.)

In [ ]:
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
# máscara de "colores saturados" (ropa llamativa) vs suelo gris
mask = cv2.inRange(hsv, (0, 80, 50), (179, 255, 255))
show(frame, hsv[:, :, 0], mask, titles=["BGR", "Canal H", "Máscara S>80"])

## 3. Filtros y convolución

Una **convolución** desliza un kernel pequeño (p. ej. 3×3) por la imagen y en cada posición calcula la suma
ponderada. Es exactamente lo que hace una capa `Conv2d` de una CNN, solo que allí los pesos del kernel
**se aprenden**.

- **Blur gaussiano**: suaviza / quita ruido.
- **Sobel**: derivada → resalta bordes en x o y.
- **Canny**: bordes finos (Gauss + Sobel + supresión de no-máximos + histéresis).
- **Morfología**: `erode` (encoge blancos), `dilate` (crece), `open` = erode+dilate (quita puntitos),
  `close` = dilate+erode (rellena huecos). Muy útil para limpiar máscaras.

In [ ]:
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (7, 7), 0)
sobel_x = cv2.convertScaleAbs(cv2.Sobel(blur, cv2.CV_16S, 1, 0, ksize=3))
edges = cv2.Canny(blur, 50, 150)
k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
show(sobel_x, edges, clean, titles=["Sobel x", "Canny", "Máscara tras OPEN"])

# Convolución "a mano" con un kernel arbitrario (lo mismo que una capa conv con pesos fijos):
kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32)  # enfocar
sharp = cv2.filter2D(frame, -1, kernel)

## 4. ✍️ EJERCICIO: *letterbox* (el preprocesado de YOLO)

YOLO trabaja con entradas cuadradas (p. ej. 640×640). Si haces `resize` directo, **deformas** a las personas
(las haces más gordas o más delgadas) y empeora la detección. Lo que se hace es **letterbox**:
1. Escalar manteniendo el aspecto: `r = min(640/H, 640/W)`.
2. Rellenar (*padding*) con gris (114) hasta 640×640, centrando la imagen.
3. Guardar `r` y el padding `(dx, dy)` para **deshacer la transformación** sobre las cajas predichas:
   `x_orig = (x_pred - dx) / r`.

Implementa `letterbox(img, size=640)` → `(img_out, r, (dx, dy))` y `unletterbox_boxes(boxes, r, dx, dy)`.

In [ ]:
# SOLUCIÓN (ábrela solo después de intentarlo)
def letterbox(img, size=640, color=(114, 114, 114)):
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nw, nh = round(w * r), round(h * r)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)  # ¡ojo! cv2.resize recibe (W, H)
    dx, dy = (size - nw) // 2, (size - nh) // 2
    out = np.full((size, size, 3), color, dtype=img.dtype)
    out[dy:dy + nh, dx:dx + nw] = resized
    return out, r, (dx, dy)

def unletterbox_boxes(boxes, r, dx, dy):
    boxes = np.asarray(boxes, dtype=np.float32).copy()
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - dx) / r
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] - dy) / r
    return boxes

In [ ]:
lb, r, (dx, dy) = letterbox(frame)
print("original", frame.shape, "→", lb.shape, "| r =", round(r, 3), "| pad =", (dx, dy))
# test: una caja en la imagen original → letterbox → vuelta
box = np.array([[100, 50, 300, 400]], np.float32)
box_lb = box * r + np.array([dx, dy, dx, dy])
assert np.allclose(unletterbox_boxes(box_lb, r, dx, dy), box, atol=1e-3)
print("✅ letterbox correcto")
show(lb, titles=["letterbox 640×640"])

## 5. Vídeo: leer, procesar, escribir

Conceptos que conviene saber explicar:
- Un vídeo es un **contenedor** (mp4, mkv) con un **stream comprimido** por un **códec** (H.264, H.265/HEVC, AV1).
- Compresión temporal: **I-frames** (keyframes, imagen completa), **P/B-frames** (solo diferencias).
  Por eso **saltar a un frame arbitrario (seek) es lento**: hay que decodificar desde el keyframe anterior.
- **Decodificar cuesta CPU.** En un pipeline en tiempo real a 1080p, la decodificación puede ser el cuello
  de botella antes que el modelo. Solución: decodificar en GPU (NVDEC, vía FFmpeg `-hwaccel cuda`,
  GStreamer o DeepStream) y no copiar frames de vuelta a CPU.
- `cap.get(cv2.CAP_PROP_FPS)` puede mentir con vídeos de *frame rate variable* (VFR, típico del móvil):
  para timestamps fiables usa `CAP_PROP_POS_MSEC` o conviértelo con FFmpeg a CFR (`-r 30`).

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS)
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
out_path = DATA / "01_salida.mp4"
writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

t0, i = time.perf_counter(), 0
while i < 150:
    ok, f = cap.read()
    if not ok:
        break
    ts = cap.get(cv2.CAP_PROP_POS_MSEC)
    cv2.putText(f, f"frame {i}  t={ts/1000:.2f}s", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    writer.write(f)
    i += 1
dt = time.perf_counter() - t0
cap.release(); writer.release()
print(f"Leídos+escritos {i} frames en {dt:.2f}s → {i/dt:.0f} FPS (solo I/O, sin modelo)")
print("Guardado en", out_path)

**Medir bien el tiempo**: separa siempre *decodificación*, *preprocesado*, *inferencia*, *postproceso*
(NMS, tracking) y *escritura*. Si hay que acelerar el pipeline, la
primera respuesta correcta es **"primero mediría dónde se va el tiempo"**.

Seek directo a un frame:

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
t0 = time.perf_counter()
cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2)
ok, mid = cap.read()
print(f"seek al frame {n//2}: {1000*(time.perf_counter()-t0):.1f} ms")
cap.release()

## 6. Sustracción de fondo (cámara fija)

Con cámara estática, **MOG2** modela cada píxel como una mezcla de gaussianas y marca como *foreground* lo que
no encaja. Barato y sin GPU. Hoy no sustituye a un detector, pero sirve para: pre-filtrar zonas con movimiento,
saltarse frames sin actividad, o detectar objetos pequeños rápidos (p. ej. un balón) que el detector pierde.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
mog = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=25, detectShadows=True)
for _ in range(80):
    ok, f = cap.read()
    fg = mog.apply(f)
cap.release()
fg = cv2.morphologyEx((fg == 255).astype(np.uint8) * 255, cv2.MORPH_OPEN, k)  # 127 = sombra, la quitamos
cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
vis = f.copy()
for c in cnts:
    if cv2.contourArea(c) > 300:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 0, 255), 2)
show(fg, vis, titles=["foreground MOG2", "blobs > 300 px"])

## 7. Flujo óptico (Lucas–Kanade) → base del *camera motion compensation*

El flujo óptico estima cuánto se mueve cada punto entre dos frames. **Lucas–Kanade** lo hace para puntos
dispersos (esquinas, `goodFeaturesToTrack`). Si la cámara se mueve (paneo en un partido), casi todos los puntos
del fondo se mueven igual → estimas una **transformación afín/homografía global** con RANSAC
y la usas para corregir la predicción del Kalman. Eso es exactamente el **GMC de BoT-SORT** (notebook 06).

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
_, f1 = cap.read()
for _ in range(5):
    _, f2 = cap.read()
cap.release()
g1, g2 = (cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in (f1, f2))
p1 = cv2.goodFeaturesToTrack(g1, maxCorners=500, qualityLevel=0.01, minDistance=10)
p2, st, _ = cv2.calcOpticalFlowPyrLK(g1, g2, p1, None)
good1, good2 = p1[st == 1], p2[st == 1]
A, inliers = cv2.estimateAffinePartial2D(good1, good2, method=cv2.RANSAC)
print("Movimiento global estimado (afín 2×3):\n", np.round(A, 3))
print("→ cámara fija: ~identidad. Inliers:", int(inliers.sum()), "/", len(inliers))

## 8. FFmpeg: chuleta

```bash
ffprobe -v error -show_streams video.mp4                    # info de streams (códec, fps, resolución)
ffmpeg -i in.mp4 -vf fps=5 frames/%05d.jpg                   # extraer 5 frames/s (para anotar)
ffmpeg -i in.mp4 -vf scale=1280:-2 -c:v libx264 -crf 23 out.mp4   # reescalar y recomprimir
ffmpeg -ss 30 -i in.mp4 -t 10 -c copy clip.mp4               # recortar SIN recodificar (rápido; corta en keyframe)
ffmpeg -hwaccel cuda -i in.mp4 ...                           # decodificar en GPU NVIDIA
ffmpeg -i rtsp://camara/stream -c copy grabacion.mp4         # grabar cámara IP
```

**GStreamer** es un framework de *pipelines* de multimedia (elementos encadenados: `rtspsrc ! decodebin !
videoconvert ! appsink`). **DeepStream** (NVIDIA) es GStreamer + plugins de inferencia TensorRT + tracker,
todo en GPU sin copias a CPU: es la forma de procesar muchas cámaras en tiempo real.